<a href="https://colab.research.google.com/github/LucySanders84/genomic-classifier-llm/blob/integrate-from-colab/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',
force_remount=True)

In [ ]:
# Remove potentially conflicting packages
!pip -q uninstall -y torch torchvision torchaudio triton transformers accelerate datasets evaluate numpy jax jaxlib

# Upgrade pip
!pip -q install --upgrade pip

# Install PyTorch CUDA build
!pip -q install --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.2.2+cu121 torchvision==0.17.2+cu121 torchaudio==2.2.2+cu121

# Install compatible stack
!pip -q install numpy==1.26.4
!pip -q install triton==2.2.0
!pip -q install transformers==4.40.2 accelerate==0.29.3 datasets==2.19.1 evaluate==0.4.2 scikit-learn



In [ ]:
# Restart runtime to remove cached dependency versions
import os
os.kill(os.getpid(), 9)

In [ ]:
import numpy as np
print("numpy:", np.__version__)
import torch
print("torch:", torch.__version__)
import transformers
print("transformers:", transformers.__version__)
import triton
print("triton:", triton.__version__)
import accelerate
print("accelerate:", accelerate.__version__)
import datasets
print("datasets:", datasets.__version__)
import evaluate
print("evaluate:", evaluate.__version__)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip -q uninstall -y peft sentence-transformers torchtune

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    set_seed
)

import evaluate
from sklearn.metrics import average_precision_score

%cd /content/drive/MyDrive/dnabert_project

SEED = 42
set_seed(SEED)

In [ ]:
MODEL = "quietflamingo/dnabert2-fixed"

MAX_LEN = 128
BATCH_TRAIN = 8
BATCH_EVAL = 16
LR = 2e-5
EPOCHS = 2

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
def load_csv(path):
    df = pd.read_csv(path)

    if "sequence" not in df.columns or "label" not in df.columns:
        raise ValueError(f"{path} must have columns sequence,label. Found: {list(df.columns)}")

    df["sequence"] = df["sequence"].astype(str).str.replace(" ", "").str.upper()
    df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)

    return Dataset.from_pandas(df, preserve_index=False)

train_ds = load_csv("train.csv")
val_ds = load_csv("val.csv")
test_ds = load_csv("test.csv")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    trust_remote_code=True
)

def tokenize(batch):
    return tokenizer(
        batch["sequence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["sequence"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["sequence"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["sequence"])

In [ ]:
train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

cols = ["input_ids", "attention_mask", "labels"]

train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

In [ ]:
from transformers.models.bert.configuration_bert import BertConfig

class DNABERT2Classifier(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()

        config = BertConfig.from_pretrained(model_name)

        self.encoder = AutoModel.from_pretrained(
            model_name,
            config=config,
            trust_remote_code=True
        )

        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        if isinstance(outputs, tuple):
            last_hidden = outputs[0]
        else:
            last_hidden = outputs.last_hidden_state

        cls_emb = last_hidden[:, 0, :]
        logits = self.classifier(self.dropout(cls_emb))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

model = DNABERT2Classifier(MODEL, num_labels=2)

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
roc_auc = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    logits = np.array(logits)
    labels = np.array(labels)

    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=1)
    pos_probs = probs[:, 1]

    out = {}

    out["accuracy"] = accuracy.compute(
        predictions=preds,
        references=labels
    )["accuracy"]

    out["f1"] = f1.compute(
        predictions=preds,
        references=labels,
        average="binary"
    )["f1"]

    try:
        out["roc_auc"] = roc_auc.compute(
            prediction_scores=pos_probs,
            references=labels
        )["roc_auc"]
    except Exception:
        out["roc_auc"] = float("nan")

    try:
        out["pr_auc"] = average_precision_score(labels, pos_probs)
    except Exception:
        out["pr_auc"] = float("nan")

    return out

In [ ]:
training_args = TrainingArguments(

    output_dir="dnabert2_output",
    seed=SEED,

    learning_rate=LR,
    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,

    gradient_accumulation_steps=2,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2,

    remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
print("Validation metrics")
print(trainer.evaluate(val_ds))

print("Test metrics")
print(trainer.evaluate(test_ds))

In [ ]:
trainer.save_model("/content/drive/MyDrive/dnabert_project/model")
tokenizer.save_pretrained("/content/drive/MyDrive/dnabert_project/model")